In [2]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
RBI_PATH = '../data/raw/RBI_BankWise(2k16-26).xlsx'
FBIL_PATH = '../data/raw/Reference_Rates.xlsx' # The new file you got
PROCESSED_PATH = '../data/processed/cleaned_fx_data.csv'

# 2. Load and Clean RBI Data
df_rbi = pd.read_excel(RBI_PATH)
df_rbi['Date'] = pd.to_datetime(df_rbi['Date'], dayfirst=True)
df_rbi = df_rbi.rename(columns={
    'USD (INR / 1 USD)': 'USD',
    'GBP (INR / 1 GBP)': 'GBP',
    'EUR (INR / 1 EUR)': 'EUR',
    'JPY (INR / 100 JPY)': 'JPY'
})
df_rbi = df_rbi[['Date', 'USD', 'GBP', 'EUR', 'JPY']].set_index('Date')

# 3. Load and Standardize FBIL Data (The Fix)
# FBIL data has headers on the 3rd row (skip 2)
df_fbil = pd.read_excel(FBIL_PATH, skiprows=2)
df_fbil['Date'] = pd.to_datetime(df_fbil['Date'], dayfirst=True)

# Map the FBIL names to our standard names
mapping = {
    'INR / 1 USD': 'USD', 'INR/1 USD': 'USD',
    'INR / 1 GBP': 'GBP', 'INR / 1 EUR': 'EUR',
    'INR / 100 JPY': 'JPY'
}
df_fbil['Currency'] = df_fbil['Currency Pairs'].map(mapping)
df_fbil = df_fbil.dropna(subset=['Currency'])

# Pivot FBIL data from "Long" format to "Wide" format (columns)
df_fbil_pivot = df_fbil.pivot_table(index='Date', columns='Currency', values='Rate')

# 4. Merge the two datasets
# This fills the NaN gaps in RBI data using the values from FBIL
df_combined = df_rbi.combine_first(df_fbil_pivot)
df_combined = df_combined.sort_index()

# 5. Handle Gaps (Weekends/Holidays)
full_range = pd.date_range(start=df_combined.index.min(), end=df_combined.index.max(), freq='D')
df_final = df_combined.reindex(full_range).ffill()

# 6. Calculate Returns and Volatility (Sprint 2 Milestone)
for cur in ['USD', 'GBP', 'EUR', 'JPY']:
    df_final[f'{cur}_Return'] = df_final[cur].pct_change()
    df_final[f'{cur}_Volatility'] = df_final[f'{cur}_Return'].rolling(window=30).std()

# 7. Save the Final "Gold Standard" CSV
os.makedirs('../data/processed', exist_ok=True)
df_final.to_csv(PROCESSED_PATH)

print("End-to-End Preprocessing Complete! RBI and FBIL data merged successfully.")
print(f"Total days covered: {len(df_final)}")

End-to-End Preprocessing Complete! RBI and FBIL data merged successfully.
Total days covered: 3654


C:\Users\DEIVANAI\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [3]:
import pandas as pd

# 1. Generate Summary Statistics
# This shows the 'Health' of your data (Mean, Min, Max)
stats_summary = df_final[['USD', 'GBP', 'EUR', 'JPY']].describe().T
stats_summary['Missing_Values'] = df_final[['USD', 'GBP', 'EUR', 'JPY']].isnull().sum()

# 2. Automated Validation Checks
# This is what you show the teacher to prove the 10-year gap is fixed
validation_report = {
    "Total Calendar Days": len(df_final),
    "Start Date": df_final.index.min().date(),
    "End Date": df_final.index.max().date(),
    "Is Date Index Continuous?": len(df_final) == (df_final.index.max() - df_final.index.min()).days + 1,
    "Any Missing Rates?": df_final[['USD', 'GBP', 'EUR', 'JPY']].isnull().values.any()
}

print("--- DATA HEALTH REPORT ---")
print(stats_summary)
print("\n--- INTEGRITY CHECKS ---")
for check, result in validation_report.items():
    print(f"{check}: {result}")

--- DATA HEALTH REPORT ---
      count       mean       std      min        25%      50%         75%  \
USD  3654.0  75.119219  7.388799  63.3482  68.583875  74.1308   82.718200   
GBP  3654.0  97.459565  9.281944  79.8563  90.475600  96.8881  103.381875   
EUR  3654.0  84.062640  7.753627  68.2503  78.408300  83.9660   89.170900   
JPY  3654.0  61.588229  4.713286  51.6200  57.870000  60.9800   65.297500   

          max  Missing_Values  
USD   91.0202               0  
GBP  121.6103               0  
EUR  106.9506               0  
JPY   72.0600               0  

--- INTEGRITY CHECKS ---
Total Calendar Days: 3654
Start Date: 2016-01-01
End Date: 2026-01-01
Is Date Index Continuous?: True
Any Missing Rates?: False


In [4]:
from statsmodels.tsa.stattools import adfuller

def perform_adf_test(series, currency_name):
    print(f"--- ADF Test for {currency_name} Returns ---")
    # Drop NaNs created by pct_change()
    result = adfuller(series.dropna())
    p_value = result[1]
    
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {p_value:.4f}")
    
    if p_value <= 0.05:
        print(f"RESULT: {currency_name} Returns are STATIONARY. (Ready for AI Modeling)\n")
    else:
        print(f"RESULT: {currency_name} Returns are NON-STATIONARY.\n")

# Run tests for all currencies
for cur in ['USD', 'GBP', 'EUR', 'JPY']:
    perform_adf_test(df_final[f'{cur}_Return'], cur)

--- ADF Test for USD Returns ---
ADF Statistic: -59.9045
p-value: 0.0000
RESULT: USD Returns are STATIONARY. (Ready for AI Modeling)

--- ADF Test for GBP Returns ---
ADF Statistic: -59.9120
p-value: 0.0000
RESULT: GBP Returns are STATIONARY. (Ready for AI Modeling)

--- ADF Test for EUR Returns ---
ADF Statistic: -59.8445
p-value: 0.0000
RESULT: EUR Returns are STATIONARY. (Ready for AI Modeling)

--- ADF Test for JPY Returns ---
ADF Statistic: -61.9588
p-value: 0.0000
RESULT: JPY Returns are STATIONARY. (Ready for AI Modeling)



In [6]:
# Calculate the 'High Risk' Threshold (90th percentile)
# This tells Kanishkhan exactly when the market is 'too nervous'
usd_risk_threshold = df_final['USD_Volatility'].quantile(0.90)

print(f"USD High-Risk Threshold: {usd_risk_threshold:.4f}")
print("Note for Risk Engine: Use this value as the boundary for 'Critical Risk' alerts.")

USD High-Risk Threshold: 0.0035
Note for Risk Engine: Use this value as the boundary for 'Critical Risk' alerts.


In [8]:
# Find the date with the highest single-day USD jump (worst day for INR)
worst_day_idx = df_final['USD_Return'].idxmax()
worst_rate_change = df_final.loc[worst_day_idx, 'USD_Return']

print(f"Worst Market Shock Date: {worst_day_idx.date()}")
print(f"Daily Percentage Jump: {worst_rate_change:.2%}")
print("Note for Modelling: Use this date for the business loss impact scenario.")

Worst Market Shock Date: 2019-08-05
Daily Percentage Jump: 1.38%
Note for Modelling: Use this date for the business loss impact scenario.
